# NuFrost Evaluation Notebook
This notebook runs the accuracy assessment for NuFrost, Zhu2015, and HANTS algorithms using simulated gap experiments.

## Configuration

In [ ]:
from pathlib import Path
import os

MOUNT_POINT_IN_COLAB = Path("/content/drive")
PROJECT_PATH_IN_GDRIVE = Path("WorkSpaces/nufrost")
PROJECT_DIR = MOUNT_POINT_IN_COLAB / "MyDrive" / PROJECT_PATH_IN_GDRIVE
IMAGE_DIR   = PROJECT_DIR / "data/hls"
OUTPUT_DIR  = PROJECT_DIR / "data/output"
CACHE_DIR   = PROJECT_DIR / "data/cache"
OUTPUT_CSV_PATH = OUTPUT_DIR / "evaluation_results_hls.csv"

# Let the script find ALL chunks across ALL coordinates and ALL bands if IMAGE_NAMES is empty.
IMAGE_NAMES = []
SAMPLE_POINTS_NUM = 50000


In [ ]:
import os
from google.colab import drive # type: ignore[import]

drive.mount(MOUNT_POINT_IN_COLAB.as_posix())
os.chdir(PROJECT_DIR)
print(f"[Working directory changed to: {os.getcwd()}]")

In [ ]:
!apt-get install -y gdal-bin
%pip install -r requirements.txt

In [ ]:
import src.data_loader
import importlib
import src.evaluation
import pandas as pd
import glob
import re
from collections import defaultdict

from config import build_args
from IPython.display import display

importlib.reload(src.nufrost)
importlib.reload(src.zhu2015)
importlib.reload(src.hants)
importlib.reload(src.evaluation)
importlib.reload(src.data_loader)

if IMAGE_NAMES:
    image_paths_list = [[(IMAGE_DIR / name).as_posix()] for name in IMAGE_NAMES]
else:
    # Auto-detect all distinct coordinates and bands, then construct their VRTs
    files = glob.glob((IMAGE_DIR / "*.tif").as_posix())
    loc_ids = set()
    for f in files:
        f_name = Path(f).name
        match = re.search(r"_([A-Z0-9]+)_lon([0-9.]+)_lat([0-9.]+).*?(?:_part\d+)?(?:-\d{10}-\d{10})?\.tif$", f_name)
        if match:
            band = match.group(1)
            lon = float(match.group(2))
            lat = float(match.group(3))
            loc_ids.add((band, lon, lat))

    image_paths_list = []
    for band, lon, lat in loc_ids:
        # find_image_chunks groups spatial tiles into VRTs correctly for each temporal part
        chunks = src.data_loader.find_image_chunks(IMAGE_DIR.as_posix(), lon, lat, band)
        if chunks:
            image_paths_list.extend(chunks)

print(f"Found {len(image_paths_list)} distinct spatial/band chunks to evaluate.")


In [ ]:
print("========== Starting Accuracy Assessment ==========")

all_results = []

for image_paths in image_paths_list:
    first_path = Path(image_paths[0])
    match = re.search(r"([A-Z0-9]+_lon[0-9.]+_lat[0-9.]+)", first_path.stem)
    loc_id = match.group(1) if match else first_path.stem
    print(f"\n--- Evaluating: {loc_id} ---")

    # 1. Initialize arguments
    args = build_args({})
    args.image = image_paths  # list of paths
    args.n_jobs = -1  # Set -1 to use all available cores in Colab
    args.cache_dir = CACHE_DIR.as_posix()  # Ensure cache directory is set for Colab

    # 2. Run evaluation
    df_results = src.evaluation.evaluate_algorithms(
        image_path=args.image,
        args=args,
        num_points=SAMPLE_POINTS_NUM
    )

    # Add image name column to distinguish results
    df_results.insert(0, "Image", loc_id)
    all_results.append(df_results)

# 3. Combine and save results
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)

    # Ensure output directory exists
    Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)

    # Save to CSV
    final_df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"\n[Success] All results saved to: {OUTPUT_CSV_PATH}")

    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    display(final_df)
else:
    print("No valid images processed.")
